In [ ]:
#Introduction To Data Ingestion
import os
from typing import List, Dict, Any
import pandas as pd

In [ ]:

import langchain
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)
print("LangChain Version:", langchain.__version__)

LangChain Version: 1.4.1


In [ ]:
##create a simple document
doc = Document(
    page_content="Hello, my dog is cute",
    metadata={"source": "myfile.txt",
              "page_number": 1,
              "author": "John Doe",
              "date_created": "2023-01-01",
              "custom_field": "custom_value"}
)
print("DocumentStructure");
print("Document:", doc);
print("Page Content:", doc.page_content);
print("Metadata:", doc.metadata);
## metadata is crucial for  filtering search  results,tracking document source,providing context in responses, Debugging and Auditing)

DocumentStructure
Document: page_content='Hello, my dog is cute' metadata={'source': 'myfile.txt', 'page_number': 1, 'author': 'John Doe', 'date_created': '2023-01-01', 'custom_field': 'custom_value'}
Page Content: Hello, my dog is cute
Metadata: {'source': 'myfile.txt', 'page_number': 1, 'author': 'John Doe', 'date_created': '2023-01-01', 'custom_field': 'custom_value'}


##text files reading(.txt)

In [ ]:
##crete a simple txt file for ingestion
import os
os.makedirs("data/text_files", exist_ok=True)


In [ ]:
sample_text = {
    "data/text_files/python_intro.txt": """Python is a high-level, interpreted programming
     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,
      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation""",
    "data/text_files/MachineLearning_intro.txt": """Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data."""
}

for filepath, content in sample_text.items():
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"Sample file created: {filepath}")

Sample file created: data/text_files/python_intro.txt
Sample file created: data/text_files/MachineLearning_intro.txt


TextLoader -Read Single File

In [ ]:
from langchain_community.document_loaders import TextLoader
##Loading thhe single text file using TextLoader
loader = TextLoader("data/text_files/python_intro.txt",encoding="utf-8")
documents = loader.load()
print("Loaded Documents:", documents);
print(type(documents))
print("Page Content:", documents[0].page_content);
print("content length:", len(documents[0].page_content));

print("Metadata:", documents[0].metadata);
print("Document Structure:", documents[0]);


Loaded Documents: [Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python is a high-level, interpreted programming\n     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,\n      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation')]
<class 'list'>
Page Content: Python is a high-level, interpreted programming
     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,
      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation
content length: 309
Metadata: {'source': 'data/text_files/python_intro.txt'}
Document Structure: page_content='Python is a high-level, interpreted programming
     language known for its clear syntax and exceptional readability. Created by Guido v

##Directory Loader-Multiple Text Files

In [ ]:


%pip install unstructured

from langchain_community.document_loaders import DirectoryLoader, TextLoader

## Loading all the text files in a directory using DirectoryLoader
directory_loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    show_progress=True,
    loader_kwargs={"encoding": "utf-8"},
)
documents = directory_loader.load()

print("Loaded Documents:", documents)
for i, doc in enumerate(documents):
    print(f"Document {i + 1}:")
    print("Page Content:", doc.page_content)
    print("Content Length:", len(doc.page_content))
    print("Metadata:", doc.metadata)
    print("Document Structure:", doc)


c:\Users\Admin\Downloads\Udemy_Python\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


100%|██████████| 2/2 [00:00<00:00, 749.52it/s]

Loaded Documents: [Document(metadata={'source': 'data\\text_files\\MachineLearning_intro.txt'}, page_content='Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data.'), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python is a high-level, interpreted programming\n     language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991,\n      it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation')]
Document 1:
Page Content: Machine learning is a branch of artificial intelligence tha

##Text spliting statergies


In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)
print("Text splitters imported successfully.")

Text splitters imported successfully.


In [ ]:
#Method 1-character Text Spliter
text=documents[0].page_content
text

'Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data.'

In [ ]:
# Method 1: Character-based splitting
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
import os

# Load documents if they were not created in a previous cell
if "documents" not in globals():
    if not os.path.exists("data/text_files"):
        os.makedirs("data/text_files", exist_ok=True)
        sample_text = {
            "data/text_files/python_intro.txt": """Python is a high-level, interpreted programming language known for its clear syntax and exceptional readability. Created by Guido van Rossum and released in 1991, it has become one of the most popular languages in the world for web development, data science, artificial intelligence, and automation""",
            "data/text_files/MachineLearning_intro.txt": """Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data."""
        }

        for filepath, content in sample_text.items():
            os.makedirs(os.path.dirname(filepath), exist_ok=True)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(content)

    directory_loader = DirectoryLoader(
        "data/text_files",
        glob="*.txt",
        loader_cls=TextLoader,
        show_progress=False,
        loader_kwargs={"encoding": "utf-8"},
    )
    documents = directory_loader.load()

if not documents:
    raise ValueError("No documents were loaded. Make sure the text files exist and the loader cell has run.")

text = documents[0].page_content
print("CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
splits = char_splitter.split_text(text)
print(f"Created {len(splits)} chuunks")
print("First Split:", splits[0][:100])
print("Split Documents:", splits)
print("Last Split:", splits[-1][:100])
print("Number of Splits:", len(splits))


CHARACTER TEXT SPLITTER
Created 1 chuunks
First Split: Machine learning is a branch of artificial intelligence that uses data and algorithms to let compute
Split Documents: ['Machine learning is a branch of artificial intelligence that uses data and algorithms to let computers learn and make predictions without hard-coded rules. What is Machine Learning? Core idea: Computers find patterns in data instead of following manual instructions. Training: Models learn by reviewing sample data and adjusting parameters. Inference: Trained models make predictions on new, unseen data.']
Last Split: Machine learning is a branch of artificial intelligence that uses data and algorithms to let compute
Number of Splits: 1


#Method 2:Recursive Character Splitting(Recomended)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

if "documents" not in globals() or not documents:
    raise ValueError("No documents were loaded. Run the text loader cell first.")

text = documents[0].page_content

print("\nRECURSIVE CHARACTER TEXT SPLITTER")
recursive_char_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len,
)
recursive_splits = recursive_char_splitter.split_text(text)
print(f"Created {len(recursive_splits)} chunks")
print("First Split:", recursive_splits[0][:100])
print(recursive_splits[0])
print("--------------------")
print(recursive_splits[1])